# Qwen Text Seg batch masks (energyx-drive)

This notebook predicts text-layer masks for all images under `INPUT_ROOT` and writes **only masks** to `OUTPUT_MASK_ROOT`.

Notes:
- Uses the full text-seg checkpoint (same as `qwen-text-seg.ipynb`).
- Keeps folder structure under the mask root to avoid filename collisions.

In [1]:
import os
from pathlib import Path

import torch
from PIL import Image
from tqdm.auto import tqdm

from diffsynth.pipelines.qwen_image import QwenImagePipeline, ModelConfig
from diffsynth import load_state_dict


def _find_repo_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / "diffsynth").exists() and (parent / "pyproject.toml").exists():
            return parent
    return start


def _resolve_model_base(repo_root: Path) -> Path:
    env = os.environ.get("DIFFSYNTH_MODEL_BASE_PATH")
    if env:
        return Path(env)
    candidates = [
        repo_root / "diffsynth_models",
        Path("/mnt/lica-data-2/for_jjseol/diffsynth_models"),
        Path("/mnt/data/for_jjseol/diffsynth_models"),
    ]
    for cand in candidates:
        if (cand / "Qwen").exists():
            return cand
    return candidates[0]


REPO_ROOT = _find_repo_root(Path(".").resolve())
DEFAULT_MODEL_BASE = _resolve_model_base(REPO_ROOT)
print(f"[model] base: {DEFAULT_MODEL_BASE}")

INPUT_ROOT = Path("/home/ubuntu/Downloads/energyx-drive")
OUTPUT_MASK_ROOT = Path("/home/ubuntu/Downloads/qwen_text/mask")
OUTPUT_MASK_ROOT.mkdir(parents=True, exist_ok=True)

PROMPT = "Extract the text layer."
SEED = 123
STEPS = 40

FULL_CKPT_DIR = REPO_ROOT / "models/train/Qwen-Image-Edit-2511_full_layered_vae_text_seg"
FULL_CKPT = FULL_CKPT_DIR / "epoch-0.safetensors"

MAX_SIDE = 1024
ROUND_TO = 32
EDIT_IMAGE_AUTO_RESIZE = False
ALPHA_THRESHOLD = 50

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}


def _resize_edit_image(image, max_side=1024, round_to=32):
    w, h = image.size
    scale = min(1.0, max_side / max(w, h))
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    if round_to:
        new_w = max(round_to, (new_w // round_to) * round_to)
        new_h = max(round_to, (new_h // round_to) * round_to)
    if (new_w, new_h) != (w, h):
        image = image.resize((new_w, new_h), Image.LANCZOS)
    return image


def _iter_images(root: Path):
    if not root.exists():
        raise FileNotFoundError(f"Input root not found: {root}")
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            yield path


[model] base: /mnt/lica-data-2/for_jjseol/diffsynth_models


In [2]:
import glob


def _model_config_local_only(local_glob: str, label: str) -> ModelConfig:
    paths = sorted(glob.glob(local_glob))
    if not paths:
        raise FileNotFoundError(f"[model] Missing local weights for {label}: {local_glob}")
    path_value = paths if len(paths) > 1 else paths[0]
    print(f"[model] Using local weights: {path_value}")
    return ModelConfig(path=path_value)


def _aux_config_local_only(local_dir: Path, label: str) -> ModelConfig:
    if not local_dir.exists():
        raise FileNotFoundError(f"[aux] Missing local assets for {label}: {local_dir}")
    print(f"[aux] Using local assets: {local_dir}")
    return ModelConfig(path=str(local_dir))


def _resolve_ckpt_path(path: Path) -> Path:
    if path.is_dir():
        candidates = sorted(path.glob("*.safetensors"), key=lambda p: p.stat().st_mtime)
        if not candidates:
            raise FileNotFoundError(f"No checkpoints found in: {path}")
        return candidates[-1]
    return path


device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.bfloat16 if device == "cuda" else torch.float32

model_base = DEFAULT_MODEL_BASE
qwen_root = model_base / "Qwen"

pipe = QwenImagePipeline.from_pretrained(
    torch_dtype=torch_dtype,
    device=device,
    model_configs=[
        _model_config_local_only(
            str(qwen_root / "Qwen-Image-Edit-2511" / "transformer" / "diffusion_pytorch_model-*.safetensors"),
            "Qwen-Image-Edit-2511 transformer",
        ),
        _model_config_local_only(
            str(qwen_root / "Qwen-Image" / "text_encoder" / "model*.safetensors"),
            "Qwen-Image text encoder",
        ),
        _model_config_local_only(
            str(qwen_root / "Qwen-Image-Layered" / "vae" / "diffusion_pytorch_model.safetensors"),
            "Qwen-Image-Layered VAE",
        ),
    ],
    tokenizer_config=None,
    processor_config=_aux_config_local_only(qwen_root / "Qwen-Image-Edit" / "processor", "Qwen-Image-Edit processor"),
)

ckpt_path = _resolve_ckpt_path(FULL_CKPT_DIR if FULL_CKPT_DIR.exists() else FULL_CKPT)
print(f"[ckpt] Using full checkpoint: {ckpt_path}")
state_dict = load_state_dict(str(ckpt_path))
pipe.dit.load_state_dict(state_dict)

[model] Using local weights: ['/mnt/lica-data-2/for_jjseol/diffsynth_models/Qwen/Qwen-Image-Edit-2511/transformer/diffusion_pytorch_model-00001-of-00005.safetensors', '/mnt/lica-data-2/for_jjseol/diffsynth_models/Qwen/Qwen-Image-Edit-2511/transformer/diffusion_pytorch_model-00002-of-00005.safetensors', '/mnt/lica-data-2/for_jjseol/diffsynth_models/Qwen/Qwen-Image-Edit-2511/transformer/diffusion_pytorch_model-00003-of-00005.safetensors', '/mnt/lica-data-2/for_jjseol/diffsynth_models/Qwen/Qwen-Image-Edit-2511/transformer/diffusion_pytorch_model-00004-of-00005.safetensors', '/mnt/lica-data-2/for_jjseol/diffsynth_models/Qwen/Qwen-Image-Edit-2511/transformer/diffusion_pytorch_model-00005-of-00005.safetensors']
[model] Using local weights: ['/mnt/lica-data-2/for_jjseol/diffsynth_models/Qwen/Qwen-Image/text_encoder/model-00001-of-00004.safetensors', '/mnt/lica-data-2/for_jjseol/diffsynth_models/Qwen/Qwen-Image/text_encoder/model-00002-of-00004.safetensors', '/mnt/lica-data-2/for_jjseol/diffsy

<All keys matched successfully>

In [3]:
image_paths = sorted(_iter_images(INPUT_ROOT))
if not image_paths:
    raise FileNotFoundError(f"No images found under: {INPUT_ROOT}")

print(f"[data] found {len(image_paths)} images")

for path in tqdm(image_paths):
    edit_image = Image.open(path).convert("RGBA")
    edit_image = _resize_edit_image(edit_image, max_side=MAX_SIDE, round_to=ROUND_TO)
    width, height = edit_image.size

    result = pipe(
        PROMPT,
        edit_image=edit_image,
        seed=SEED,
        num_inference_steps=STEPS,
        height=height,
        width=width,
        edit_image_auto_resize=EDIT_IMAGE_AUTO_RESIZE,
        zero_cond_t=True,
    )

    output_rgba = result.convert("RGBA")
    alpha = output_rgba.getchannel("A")
    mask = alpha.point(lambda v: 255 if v > ALPHA_THRESHOLD else 0)

    rel = path.relative_to(INPUT_ROOT)
    out_dir = OUTPUT_MASK_ROOT / rel.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{path.stem}_mask.png"
    mask.save(out_path)

print(f"[done] masks saved under: {OUTPUT_MASK_ROOT}")

[data] found 41 images


  0%|          | 0/41 [00:00<?, ?it/s]

100%|██████████| 40/40 [00:23<00:00,  1.67it/s]

[done] masks saved under: /home/ubuntu/Downloads/qwen_text/mask


In [4]:
# Visualization (post-process only, no inference)
VIS_ROOT = Path("/home/ubuntu/Downloads/qwen_text/visualization")
VIS_ROOT.mkdir(parents=True, exist_ok=True)

image_paths = sorted(_iter_images(INPUT_ROOT))
missing_masks = []

for path in tqdm(image_paths, desc="visualize"):
    rel = path.relative_to(INPUT_ROOT)
    mask_path = OUTPUT_MASK_ROOT / rel.parent / f"{path.stem}_mask.png"
    if not mask_path.exists():
        missing_masks.append(rel.as_posix())
        continue

    img = Image.open(path).convert("RGB")
    mask = Image.open(mask_path).convert("L")
    if img.size != mask.size:
        img = img.resize(mask.size, Image.LANCZOS)

    mask_rgb = mask.convert("RGB")
    canvas = Image.new("RGB", (img.width * 2, img.height), (0, 0, 0))
    canvas.paste(img, (0, 0))
    canvas.paste(mask_rgb, (img.width, 0))

    out_dir = VIS_ROOT / rel.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{path.stem}_viz.png"
    canvas.save(out_path)

print(f"[done] visualizations saved under: {VIS_ROOT}")
if missing_masks:
    print(f"[warn] missing masks: {len(missing_masks)} (showing up to 10)")
    print("  " + "\n  ".join(missing_masks[:10]))

visualize:   0%|          | 0/41 [00:00<?, ?it/s]

[done] visualizations saved under: /home/ubuntu/Downloads/qwen_text/visualization
